In [1]:
# import all the necessary libraries

# accepting your openai key
import getpass

# import environment settings
import os
from dotenv import load_dotenv

from IPython.display import display, Markdown

# connecting to openai
from openai import OpenAI

In [ ]:
# connect to the openai library
api_key = getpass.getpass('Enter your API key: ')
os.environ['OPENAI_API_KEY'] = api_key
load_dotenv()

# initialization of the openai function
openai = OpenAI()

In [ ]:
# We wanted to scrape a website given a website url - get its title and content and use it for summarization or brouchure creation
import requests
from bs4 import BeautifulSoup

class Website:
  """
  A utility class to represent a website that we need to scrape and get its contents
  """

  def __init__(self, url: str):
    self.url = url
    # read the website using the requests library
    response = requests.get(url)
    self.body = response.content

    # parse the content of the website
    soup = BeautifulSoup(self.body, 'html.parser')

    self.title = soup.title.string
    self.text = soup.get_text(separator="\n", strip=True)
    self.links = [link.get('href') for link in soup.find_all('a')]

  def get_contents(self):
    return f"Website Title is: \n {self.title} \n \n Website Content is: \n {self.text}"
#

In [ ]:
huggingface = Website("https://huggingface.co/")
huggingface.links
#

In [ ]:
# First use of LLM is to look at the links and only select important links for our brouchure

link_system_prompt = "You are provided with a list of links found on a website. \
You are able to decide which links would be most relavant to include in the brouchure about the company, \
Include links such as About page, Company Vision and Mission, Careers page, Docs etc. \n"

link_system_prompt += " You should respond in JSON as below example:"
link_system_prompt += """
{
    "links": [
      {"type": "About", "url": "https://yourwebsite.url/about"},
      {"type": "Vision", "url": "https://yourwebsite.url/vision"},
      {"type": "Mission", "url": "https://full.url/mission"},
      {"type": "Careers", "url": "https://yourwebsite.url/careers"},
      {"type": "Docs", "url": "https://another.full.url/docs"}
    ]
}
"""

In [ ]:
print(link_system_prompt)

In [ ]:
# user prompt for getting relavant links

def get_links_user_prompt(website):
  user_prompt = f"Here is the list of links on the website of {website.url} - "
  user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
  user_prompt += "Links (some might be relative links):\n"
  user_prompt += "\n".join(website.links)
  return user_prompt

In [ ]:
import json

def get_links(url):
  website = Website(url)
  user_prompt = get_links_user_prompt(website)

  response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
      {"role": "system", "content": link_system_prompt},
      {"role": "user", "content": user_prompt}
    ],
    response_format={"type": "json_object"}
  )
  result = response.choices[0].message.content
  return json.loads(result)

In [ ]:
get_links("https://huggingface.co")

In [ ]:
# next step - get all the content of these links
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    # print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [ ]:
print(get_all_details("https://huggingface.co"))

In [ ]:
system_prompt = """
You are an assistant that analyzes multiple pages from a company website and creates a short, structured brochure.

Rules:
- Use only the provided website content. Do not invent facts.
- If information is missing, omit it.
- Write in clear markdown.

Output format:

## Company Overview
...

## Products / Services
...

## Company Culture
...

## Customers
...

## Careers
...

Keep it concise and factual.
"""

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Huggingface","https://huggingface.co/")